# Что меняется между вдохом и выдохом: осмотр данных

Ноутбук существует потому, что три ошибки подряд прошли все автоматические проверки и
были видны только глазами. Он не считает результат — он показывает, можно ли верить
тому, что считает конвейер.

Порядок такой: сначала проверяется, правильно ли поняты оси тома, потом — правильно ли
выделены тело и лёгкое, и только потом смотрится, что изменилось между фазами.

Данные: пары DIR-Lab COPDgene (`iBHCT` / `eBHCT`), единственный набор на диске с полным
торсом и обеими фазами.

**Техническое воспроизведение, 14.09.2026.** Ноутбук сохраняет историческую постановку и ограничения. Повторное исполнение проверяет расчёты и отображение; оно не меняет научный статус ветки. Тяжёлые результаты читаются из сохранённых пакетов с исходными проверками целостности. Источники и конфигурации используются из этой папки проекта.

In [1]:
# Portable execution support; scientific sources remain in this checkout.
import os
import sys
from pathlib import Path
_start = Path(os.environ.get("BREATH_NOTEBOOK_DIR", Path.cwd())).resolve()
_support = next((folder for parent in (_start, *_start.parents)
                 for folder in (parent, parent / "notebooks", parent / "breath geometry/notebooks")
                 if (folder / "execution_support.py").is_file()), None)
if _support is None:
    raise FileNotFoundError("Не найден notebooks/execution_support.py")
sys.path.insert(0, str(_support))
from execution_support import roots, copdgene_case_dir, lungct_input
REPO_ROOT, ARTIFACT_ROOT = roots()
repo_root = REPO_ROOT


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage

from breathgeom.io.dirlab import load_copdgene
from breathgeom.measure.wall import AIR_HU, Side, WallParams, body_mask, lung_mask

# Каталог с распакованными случаями DIR-Lab: copd1/copd1_iBHCT.img и т.д.
CASE = "copd1"

MP = WallParams()
plt.rcParams["figure.dpi"] = 110

## 1. Загрузка и отчёт об ориентации

Формат DIR-Lab безголовый: ни размеров, ни шага, ни ориентации, ни шкалы интенсивностей.
Всё это либо берётся из опубликованной таблицы, либо восстанавливается из самого
изображения — и тогда обязано быть предъявлено, а не принято на веру.

Ориентация определяется **один раз на человека** по более наполненному скану и
применяется к обеим фазам: две фазы одного человека не могут иметь разную анатомию.

In [3]:
folder = copdgene_case_dir(CASE, ARTIFACT_ROOT)
_, _, orientation = load_copdgene(folder / f"{CASE}_iBHCT.img", CASE)

volumes = {}
for tag in ("iBHCT", "eBHCT"):
    volume, spacing, _ = load_copdgene(
        folder / f"{CASE}_{tag}.img", CASE, orientation=orientation
    )
    volumes[tag] = volume
sx, sy, sz = spacing

print(f"{CASE}: {volumes['iBHCT'].shape}, шаг {sx} x {sy} x {sz} мм")
print()
print(f"позвоночник смещён на {orientation.spine_offset_px:+.0f} px -> "
      f"задняя сторона у больших индексов: {orientation.posterior_is_high_row}")
print(f"асимметрия воздуха {orientation.lung_asymmetry:+.3f} -> "
      f"правая сторона у больших индексов: {orientation.right_is_high_index}")
print(f"каудальный избыток воздуха {orientation.caudal_air_excess:+.3f}, "
      f"резкость диафрагмы {orientation.diaphragm_sharpness_ratio:.2f} -> "
      f"голова у больших индексов: {orientation.superior_is_high_index}")
print(f"признаки головы/ног согласованы: {orientation.sides_agree}")
print(f"пик воздуха восстановлен на {orientation.air_peak_hu:.0f} HU")

KeyError: 'dataset_id'

## 2. Проверка осей по анатомии

Это главная проверка ноутбука. Оси не подписаны в файле, и ошибка в них не вызывает
никакого исключения — она просто делает все дальнейшие числа бессмысленными.

Два независимых признака:

- **позвонок** — самая плотная структура, лежит сзади и по средней линии. По
  передне-задней оси его гистограмма обязана быть смещена к одному краю, по
  лево-правой — иметь пик в центре;
- **сердце** смещает левое лёгкое, поэтому по лево-правой оси воздух распределён
  асимметрично, а по передне-задней — примерно симметрично.

Если картина обратная, оси перепутаны.

In [ ]:
volume = volumes["iBHCT"]
middle = volume.shape[2] // 2
slice_hu = volume[:, :, middle].astype(np.int32)
body = body_mask(slice_hu, MP)
bone = body & (slice_hu > 300)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for a, axis, name in ((ax[0], 0, "ось 0 (ожидается: перёд-зад)"),
                      (ax[1], 1, "ось 1 (ожидается: лево-право)")):
    counts = bone.sum(axis=1 - axis)
    a.plot(counts)
    a.set_title(f"кость вдоль {name}")
    a.set_xlabel("индекс вокселя")
    a.grid(alpha=0.3)
fig.tight_layout()
plt.show()

air = body & (slice_hu < AIR_HU)
rows, columns = np.where(air)
for axis, values, name in ((0, rows, "ось 0"), (1, columns, "ось 1")):
    spanned = np.flatnonzero(body.any(axis=1 - axis))
    midline = 0.5 * (spanned[0] + spanned[-1])
    low = int((values < midline).sum())
    high = int((values >= midline).sum())
    print(f"воздух по {name}: {low} / {high}, отношение {high / max(low, 1):.2f}")

Ожидаемое для корректных осей: по **лево-правой** оси кость даёт узкий пик в центре
(позвонок на средней линии), по **передне-задней** — смещение к одному краю. Воздух
асимметричен по лево-правой оси и симметричен по передне-задней.

## 3. Профили по срезам

Три величины, каждая ловит свой класс ошибок:

- **сечение тела** — меняется ли поза и укладка между фазами;
- **сечение правого лёгкого** — работает ли выделение лёгкого вообще;
- **наружная граница сбоку** — двигается ли кожа.

Скачки сечения лёгкого между соседними срезами анатомически невозможны. Если они есть,
маска выбирает разные объекты, и всё, что построено на ней, недействительно.

In [ ]:
def profiles(volume):
    body_area, lung_area, lateral = [], [], []
    for index in range(volume.shape[2]):
        current = volume[:, :, index]
        mask = body_mask(current, MP)
        if mask is None:
            body_area.append(0.0)
            lung_area.append(0.0)
            lateral.append(np.nan)
            continue
        body_area.append(float(mask.sum()) * sx * sy / 100.0)
        lung = lung_mask(current, mask, Side.RIGHT, MP)
        lung_area.append(float(lung.sum()) * sx * sy / 100.0 if lung is not None else 0.0)
        occupied = np.where(mask.any(axis=1))[0]
        lateral.append(float(occupied.max()) * sx)
    return map(np.array, (body_area, lung_area, lateral))


body_in, lung_in, lat_in = profiles(volumes["iBHCT"])
body_ex, lung_ex, lat_ex = profiles(volumes["eBHCT"])
z = np.arange(volumes["iBHCT"].shape[2]) * sz

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
panels = (
    (body_in, body_ex, "сечение тела, см²"),
    (lung_in, lung_ex, "сечение правого лёгкого, см²"),
    (lat_in, lat_ex, "наружная граница сбоку, мм"),
)
for a, (vi, ve, title) in zip(ax, panels, strict=True):
    a.plot(z, vi, label="вдох")
    a.plot(z, ve, label="выдох")
    a.set_title(title)
    a.set_xlabel("положение среза, мм")
    a.legend()
    a.grid(alpha=0.3)
fig.tight_layout()
plt.show()

jump_in = np.abs(np.diff(lung_in)).max()
jump_ex = np.abs(np.diff(lung_ex)).max()
print(f"наибольший скачок сечения лёгкого между соседними срезами: "
      f"вдох {jump_in:.0f} см², выдох {jump_ex:.0f} см²")
print("Скачок в десятки см² означает, что маска переключается между разными объектами.")

## 4. Сколько компонент видит маска лёгкого

Правило «компонента с наибольшим центроидом по лево-правой оси = правое лёгкое» ломается,
когда лёгкие соединены: тогда возвращаются оба сразу. Здесь видно, на каких срезах это
происходит.

In [ ]:
volume = volumes["iBHCT"]
rows = []
for index in range(0, volume.shape[2], 4):
    current = volume[:, :, index]
    mask = body_mask(current, MP)
    if mask is None:
        continue
    air = ndimage.binary_opening(mask & (current < AIR_HU), np.ones((3, 3), dtype=bool))
    labels, count = ndimage.label(air)
    if count == 0:
        continue
    sizes = ndimage.sum(air, labels, range(1, count + 1))
    kept = [size for size in sizes if size > MP.min_lung_component_px]
    chosen = lung_mask(current, mask, Side.RIGHT, MP)
    rows.append((
        index,
        len(kept),
        float(sum(kept)) * sx * sy / 100.0,
        float(chosen.sum()) * sx * sy / 100.0 if chosen is not None else 0.0,
    ))

print(f"{'срез':>5} {'компонент':>10} {'весь воздух':>12} {'выбрано':>9}  диагноз")
for index, count, total, picked in rows:
    if picked == 0.0:
        verdict = "ничего не выбрано"
    elif picked > 0.75 * total and count == 1:
        verdict = "ОБА ЛЁГКИХ СЛИТЫ В ОДНО"
    elif picked < 0.2 * total:
        verdict = "выбран посторонний объект"
    else:
        verdict = "похоже на одно лёгкое"
    print(f"{index:>5} {count:>10} {total:>12.0f} {picked:>9.0f}  {verdict}")

---

## 5. Совмещение фаз

Всё дальнейшее сравнивает две фазы и потому требует общей системы координат. Совмещение
жёсткое и опирается на **позвоночник**: рёбра поворачиваются при каждом вдохе, то есть
являются частью изучаемого движения, и метрика с ними поглотила бы часть эффекта в
трансформацию.

Перекрытие столбов по Дайсу — единственная величина здесь, у которой есть смысл сама по
себе. Мера взаимной информации говорит, насколько оптимизатор доволен собой; Дайс говорит,
совпала ли кость, которая обязана была совпасть.

In [ ]:
from breathgeom.measure.align import rigid_align
from breathgeom.measure.layers import LayerParams, measure_layers
from breathgeom.measure.wall import anatomical_midline

inhale = volumes["iBHCT"]
exhale, alignment = rigid_align(inhale, volumes["eBHCT"], spacing)
MID = anatomical_midline(inhale, MP)
PHASES = (("вдох", inhale, "tab:blue"), ("выдох", exhale, "tab:orange"))

print(f"сдвиг {alignment.shift_mm:.2f} мм "
      f"{[round(v, 2) for v in alignment.translation_mm]}")
print(f"поворот, град {[round(v, 2) for v in alignment.rotation_deg]}")
print(f"перекрытие позвоночников по Дайсу: "
      f"{alignment.column_dice_before:.3f} -> {alignment.column_dice_after:.3f}")
if alignment.column_dice_after < 0.7:
    print("\nВНИМАНИЕ: половинное перекрытие тонкой структуры означает остаточное")
    print("рассогласование порядка пары миллиметров — на грани измеряемой величины.")

## 6. Границы на обеих подложках

Один и тот же набор контуров рисуется дважды: сверху на изображении вдоха, снизу на
изображении выдоха. Контур, который на своей подложке лежит по краю ткани, а на чужой уходит
внутрь или наружу, и есть искомое смещение, увиденное без единого числа.

In [ ]:
def contours(axis, index):
    for _, volume, colour in PHASES:
        current = volume[:, :, index]
        mask = body_mask(current, MP)
        if mask is None:
            continue
        axis.contour(mask.T, levels=[0.5], colors=colour, linewidths=1.1)
        lung = lung_mask(current, mask, Side.RIGHT, MP, MID)
        if lung is not None:
            axis.contour(lung.T, levels=[0.5], colors=colour, linewidths=1.1,
                         linestyles="--")


def has_lung(volume, index):
    body = body_mask(volume[:, :, index], MP)
    if body is None:
        return False
    lung = lung_mask(volume[:, :, index], body, Side.RIGHT, MP, MID)
    return lung is not None and float(lung.sum()) * sx * sy / 100.0 > 20


shared = [k for k in range(inhale.shape[2])
          if has_lung(inhale, k) and has_lung(exhale, k)]
levels = np.linspace(shared[0], shared[-1], 6)[1:5].astype(int)
print(f"срезов с лёгким в обеих фазах: {len(shared)} "
      f"({shared[0]}..{shared[-1]}, {(shared[-1] - shared[0]) * sz:.0f} мм)")

fig, axes = plt.subplots(2, 4, figsize=(17, 9))
for row, (name, volume, _) in enumerate(PHASES):
    for axis, index in zip(axes[row], levels, strict=True):
        axis.imshow(volume[:, :, index].T, cmap="gray", vmin=-1000, vmax=200,
                    origin="lower")
        contours(axis, index)
        axis.set_title(f"срез {index}, подложка — {name}", fontsize=9)
        axis.axis("off")
fig.suptitle("сплошная — тело, пунктир — правое лёгкое; синий вдох, оранжевый выдох")
fig.tight_layout()
plt.show()

## 7. Где именно измеряется

Тело целиком и плоскости, отмечающие взятые срезы. Оранжевые — границы пояса, то есть
крайние срезы, где лёгкое есть **в обеих** фазах; синие — те четыре, что показаны выше
картинками.

Пояс отсчитывается не от края тома, а от лёгкого, и намеренно не включает рёберно-
диафрагмальный синус: там лёгкое к боковой стенке не прилежит, и на выдохе его место
занимает диафрагма с печенью. Измерение в синусе даёт вошедшую печень, а не изменение
стенки.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
import trimesh
from skimage import measure as skmeasure

# Один "notebook_connected" отдаёт только text/html, а JupyterLab и VS Code не
# исполняют <script> в HTML-выводе, поэтому на месте рисунка остаётся пустота.
# Пара с plotly_mimetype добавляет application/vnd.plotly.v1+json, который они
# рисуют сами. Каждый рисунок дополнительно сохраняется отдельным html: он
# открывается любым браузером и не зависит ни от версии Lab, ни от расширений.
pio.renderers.default = "plotly_mimetype+notebook_connected"
FIGURES = REPO_ROOT / "notebooks/figures"
FIGURES.mkdir(parents=True, exist_ok=True)


def show(figure, name):
    """Показать в ноутбуке и сохранить отдельным интерактивным файлом."""
    path = FIGURES / f"{name}.html"
    figure.write_html(path, include_plotlyjs="cdn", full_html=True)
    print(f"интерактивная копия: {path.as_posix()}")
    figure.show()


# Выборка изотропная: шаг подбирается так, чтобы он был одинаков в
# миллиметрах по всем осям. Одинаковый шаг в вокселях даёт у copd1 элемент
# 5 x 5 x 20 мм, потому что срезы вчетверо толще пикселя, и поверхность
# рассыпается на вертикальные ступени. По самой грубой оси не прореживаем
# вовсе — там и так нет запаса.
STEPS = (max(int(round(sz / sx)), 1), max(int(round(sz / sy)), 1), 1)
STEP_MM = (sx * STEPS[0], sy * STEPS[1], sz * STEPS[2])
SMOOTH = 0.8


def mesh_of(mask, target_faces):
    """Изотропная поверхность маски, упрощённая до заданного числа граней.

    Размер файла ограничивается упрощением уже готовой сетки, а не
    прореживанием вокселей: квадрическое упрощение убирает грани там, где
    поверхность плоская, и сохраняет там, где она изогнута, тогда как
    прореживание вокселей огрубляет всё одинаково и создаёт ступени.
    """
    small = mask[::STEPS[0], ::STEPS[1], ::STEPS[2]]
    if small.sum() < 100:
        return None
    field = ndimage.gaussian_filter(small.astype(np.float32), SMOOTH)
    if field.max() <= 0.5:
        return None
    vertices, triangles, _, _ = skmeasure.marching_cubes(
        field, level=0.5, spacing=STEP_MM)
    if len(triangles) > target_faces:
        simplified = trimesh.Trimesh(vertices, triangles, process=False)
        simplified = simplified.simplify_quadric_decimation(face_count=target_faces)
        vertices, triangles = simplified.vertices, simplified.faces
    return np.round(np.asarray(vertices), 1), np.asarray(triangles)


body_volume = np.zeros(inhale.shape, dtype=bool)
for index in range(inhale.shape[2]):
    mask = body_mask(inhale[:, :, index], MP)
    if mask is not None:
        body_volume[:, :, index] = mask

verts, faces = mesh_of(body_volume, target_faces=60000)
print(f"шаг выборки {STEP_MM[0]:.2f} x {STEP_MM[1]:.2f} x {STEP_MM[2]:.2f} мм")
print(f"поверхность тела: {len(verts)} вершин, {len(faces)} граней")

figure = go.Figure()
figure.add_trace(go.Mesh3d(
    x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    color="#B4B2A9", opacity=0.18, name="тело на вдохе",
    showlegend=True, hoverinfo="name"))

span_x = (verts[:, 0].min(), verts[:, 0].max())
span_y = (verts[:, 1].min(), verts[:, 1].max())
for index, colour, label in (
    [(shared[0], "#D85A30", "низ пояса"), (shared[-1], "#D85A30", "верх пояса")]
    + [(int(k), "#378ADD", f"срез {int(k)}") for k in levels]
):
    height = index * sz
    figure.add_trace(go.Surface(
        x=np.array([[span_x[0], span_x[1]], [span_x[0], span_x[1]]]),
        y=np.array([[span_y[0], span_y[0]], [span_y[1], span_y[1]]]),
        z=np.full((2, 2), height),
        showscale=False, opacity=0.35,
        colorscale=[[0, colour], [1, colour]],
        name=label, showlegend=True, hoverinfo="name"))

figure.update_layout(
    title=f"{CASE}: пояс {shared[0]}..{shared[-1]} "
          f"({(shared[-1] - shared[0]) * sz:.0f} мм) и показанные срезы",
    scene=dict(xaxis_title="R, мм", yaxis_title="A, мм", zaxis_title="S, мм",
               aspectmode="data"),
    height=700, margin=dict(l=0, r=0, t=40, b=0))
show(figure, "01_body_and_slices")

## 8. Поле смещений: карта, а не кривая

Кривая по одному срезу показывает слишком мало. Здесь то же самое считается по всем срезам
с лёгким и по всем углам правого бока, и выкладывается картой «угол × уровень».

Три первые панели — смещение кожи, наружной поверхности кости и поверхности лёгкого.
Четвёртая — то, ради чего всё затевалось: изменение расстояния от кожи до лёгкого.

Смотреть надо не на медиану под панелью, а на саму карту, и прежде всего на её **нижние
строки**. Там рёберно-диафрагмальный синус: лёгкое к боковой стенке не прилежит, и на выдохе
его место занимает поднявшаяся диафрагма. Эти строки и делают медиану четвёртой панели
заметно положительной, тогда как выше по грудной клетке она близка к нулю. Именно поэтому
измерять у основания нельзя, и именно это различие невозможно увидеть в одном числе.

In [ ]:
ANGLES = np.radians(np.linspace(-70, 70, 141))
RADII = np.arange(1, 520) * 0.5


def radial_map(volume):
    """Радиусы кожи, кости и лёгкого по углам и срезам, мм."""
    out = {key: np.full((len(shared), len(ANGLES)), np.nan) for key in
           ("кожа", "кость", "лёгкое")}
    for row, index in enumerate(shared):
        current = volume[:, :, index]
        body = body_mask(current, MP)
        if body is None:
            continue
        lung = lung_mask(current, body, Side.RIGHT, MP, MID)
        centre = ndimage.center_of_mass(body)
        for column, angle in enumerate(ANGLES):
            rows = np.clip(np.round(centre[0] + RADII * np.cos(angle) / sx).astype(int),
                           0, current.shape[0] - 1)
            cols = np.clip(np.round(centre[1] + RADII * np.sin(angle) / sy).astype(int),
                           0, current.shape[1] - 1)
            along = body[rows, cols]
            if along.any():
                out["кожа"][row, column] = RADII[np.flatnonzero(along)[-1]]
            bone = along & (current[rows, cols] > 200)
            if bone.any():
                out["кость"][row, column] = RADII[np.flatnonzero(bone)[-1]]
            if lung is not None:
                inside = lung[rows, cols]
                if inside.any():
                    out["лёгкое"][row, column] = RADII[np.flatnonzero(inside)[-1]]
    return out


maps = {name: radial_map(volume) for name, volume, _ in PHASES}
extent = [np.degrees(ANGLES[0]), np.degrees(ANGLES[-1]),
          shared[-1] * sz, shared[0] * sz]

panels = [
    ("кожа", maps["выдох"]["кожа"] - maps["вдох"]["кожа"]),
    ("кость", maps["выдох"]["кость"] - maps["вдох"]["кость"]),
    ("лёгкое", maps["выдох"]["лёгкое"] - maps["вдох"]["лёгкое"]),
    ("толщина кожа−лёгкое",
     (maps["выдох"]["кожа"] - maps["выдох"]["лёгкое"])
     - (maps["вдох"]["кожа"] - maps["вдох"]["лёгкое"])),
]
limit = np.nanpercentile(np.abs(np.concatenate([p[1].ravel() for p in panels])), 98)

fig, axes = plt.subplots(1, 4, figsize=(18, 5.2))
for axis, (title, field) in zip(axes, panels, strict=True):
    image = axis.imshow(field, cmap="RdBu_r", vmin=-limit, vmax=limit,
                        extent=extent, aspect="auto")
    axis.set_title(f"{title}\nмедиана {np.nanmedian(field):+.1f} мм")
    axis.set_xlabel("угол, ° (0 = вправо)")
    axes[0].set_ylabel("уровень, мм")
    fig.colorbar(image, ax=axis, fraction=0.046)
fig.suptitle("выдох − вдох, мм; красное — наружу, синее — внутрь")
fig.tight_layout()
plt.show()

for title, field in panels:
    values = field[np.isfinite(field)]
    print(f"{title:>20}: медиана {np.median(values):+6.2f} мм, "
          f"p10 {np.percentile(values, 10):+6.2f}, p90 {np.percentile(values, 90):+6.2f}")

## 9. Экспертное соответствие: 300 размеченных пар

Соответствие точек у нас **есть**, и его не нужно восстанавливать регистрацией: DIR-Lab даёт
по 300 пар экспертных ориентиров на случай, вдох и выдох. Это прямое измерение перемещения
конкретной анатомической точки.

Ниже они пересчитаны в миллиметры и сверены с опубликованным средним смещением набора. Это
сквозная проверка всей координатной цепочки — размеров, шага, порядка осей, базы индексации,
— и единственная во всей работе, у которой ответ известен заранее.

Ограничение: ориентиры лежат **внутри лёгкого**, на сосудистых бифуркациях. Для стенки
размеченного соответствия нет ни в одном наборе.

In [ ]:
# Опубликовано на странице DIR-Lab: среднее (ско) смещение полного набора признаков.
PUBLISHED = {"copd1": 25.90, "copd2": 21.77, "copd3": 12.29, "copd4": 30.90,
             "copd5": 30.90, "copd6": 28.32, "copd7": 21.66, "copd8": 25.57,
             "copd9": 14.84, "copd10": 22.48}


def to_ras(points):
    """Воксельные индексы (x, y, z) файла -> индексы тома в RAS+."""
    x, y, z = points[:, 0].copy(), points[:, 1].copy(), points[:, 2].copy()
    if not orientation.right_is_high_index:
        x = (volumes["iBHCT"].shape[0] - 1) - x
    if orientation.posterior_is_high_row:
        y = (volumes["iBHCT"].shape[1] - 1) - y
    if not orientation.superior_is_high_index:
        z = (volumes["iBHCT"].shape[2] - 1) - z
    return np.stack([x, y, z], axis=1)


raw_in = np.loadtxt(folder / f"{CASE}_300_iBH_xyz_r1.txt") - 1.0
raw_ex = np.loadtxt(folder / f"{CASE}_300_eBH_xyz_r1.txt") - 1.0
scale = np.array([sx, sy, sz])
mm_in = to_ras(raw_in) * scale
mm_ex = to_ras(raw_ex) * scale
delta = mm_ex - mm_in
distance = np.linalg.norm(delta, axis=1)

print(f"смещение ориентиров: среднее {distance.mean():.2f} мм, "
      f"медиана {np.median(distance):.2f}, максимум {distance.max():.2f}")
print(f"опубликовано DIR-Lab для {CASE}: {PUBLISHED[CASE]:.2f} мм "
      f"(расхождение {distance.mean() - PUBLISHED[CASE]:+.2f})")
print(f"по осям, средний модуль: R {np.abs(delta[:, 0]).mean():.1f} / "
      f"A {np.abs(delta[:, 1]).mean():.1f} / S {np.abs(delta[:, 2]).mean():.1f} мм")

Ниже соответствие показано как есть: каждая пара соединена отрезком, цвет — величина
перемещения. Модель вращается мышью.

In [ ]:
lines_x, lines_y, lines_z = [], [], []
for start, stop in zip(mm_in, mm_ex, strict=True):
    lines_x += [start[0], stop[0], None]
    lines_y += [start[1], stop[1], None]
    lines_z += [start[2], stop[2], None]

figure = go.Figure()
figure.add_trace(go.Scatter3d(
    x=lines_x, y=lines_y, z=lines_z, mode="lines",
    line=dict(color="grey", width=2), name="перемещение", hoverinfo="skip"))
figure.add_trace(go.Scatter3d(
    x=mm_in[:, 0], y=mm_in[:, 1], z=mm_in[:, 2], mode="markers",
    marker=dict(size=3, color=distance, colorscale="Turbo",
                colorbar=dict(title="мм")),
    name="вдох",
    text=[f"{value:.1f} мм" for value in distance], hoverinfo="text"))
figure.add_trace(go.Scatter3d(
    x=mm_ex[:, 0], y=mm_ex[:, 1], z=mm_ex[:, 2], mode="markers",
    marker=dict(size=3, color="black", opacity=0.45), name="выдох",
    hoverinfo="skip"))
figure.update_layout(
    title=f"{CASE}: 300 экспертных пар, среднее смещение {distance.mean():.1f} мм",
    scene=dict(xaxis_title="R, мм", yaxis_title="A, мм", zaxis_title="S, мм",
               aspectmode="data"),
    height=650, margin=dict(l=0, r=0, t=40, b=0))
show(figure, "02_expert_landmarks")

Соответствие даёт не только картинку, но и **единственный внешний критерий качества поля
смещений**, какой в этой работе есть: TRE, среднее расстояние между парными ориентирами
после того как поле применено.

Ниже он посчитан для жёсткого совмещения. Ожидать от него многого нельзя и не нужно: жёсткое
преобразование по построению не описывает деформацию, а ориентиры лежат внутри лёгкого,
которое деформируется сильнее всего. Число важно как **линейка**, а не как результат.

In [ ]:
after = np.linalg.norm(alignment.to_fixed(mm_ex) - mm_in, axis=1)
print(f"TRE до совмещения:    {distance.mean():6.2f} мм")
print(f"TRE после жёсткого:   {after.mean():6.2f} мм  "
      f"(убрано {100 * (1 - after.mean() / distance.mean()):.0f}%)")
print()
print("Опубликованный уровень деформируемых методов на DIR-Lab COPDgene — около 1 мм.")
print("Жёсткое совмещение снимает позу, а она здесь мала; движение лёгкого остаётся")
print("целиком. Поля смещений у нас, следовательно, ещё нет — есть линейка для него.")

## 10. Компартменты в объёме, вращаемые

Четыре ткани, обе фазы, наложенные друг на друга. Маски строятся по тем же окнам Хаунсфилда,
что и все измерения, и только внутри пояса срезов с лёгким в обеих фазах.

Модели интерактивные: вращать мышью, легенду можно переключать по слоям.

In [ ]:
BAND = (shared[0], shared[-1])

COMPARTMENTS = {
    "лёгкие": lambda hu, body: body & (hu < -400),
    "кости": lambda hu, body: body & (hu > 200),
    "мышца": lambda hu, body: body & (hu >= -29) & (hu <= 150),
    "жир": lambda hu, body: body & (hu >= -190) & (hu <= -30),
}
COLOURS = {"вдох": "#378ADD", "выдох": "#D85A30"}


def compartment(volume, rule):
    mask = np.zeros(volume.shape, dtype=bool)
    for index in range(BAND[0], BAND[1] + 1):
        current = volume[:, :, index]
        body = body_mask(current, MP)
        if body is None:
            continue
        mask[:, :, index] = rule(current, body)
    return mask


masks = {name: {phase: compartment(volume, rule) for phase, volume, _ in PHASES}
         for name, rule in COMPARTMENTS.items()}


def surface(mask, colour, label, visible):
    built = mesh_of(mask, target_faces=25000)
    if built is None:
        return None
    verts, faces = built
    return go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=colour, opacity=0.35, name=label, showlegend=True,
        visible=visible, hoverinfo="name")


figure = go.Figure()
for tissue, per_phase in masks.items():
    for phase, mask in per_phase.items():
        mesh = surface(mask, COLOURS[phase], f"{tissue}, {phase}",
                       True if tissue == "лёгкие" else "legendonly")
        if mesh is not None:
            figure.add_trace(mesh)
print(f"вершин во всех поверхностях: "
      f"{sum(len(trace.x) for trace in figure.data):,}")
figure.update_layout(
    title="компартменты, вдох синим и выдох оранжевым "
          "(нажатие в легенде включает и выключает слой)",
    scene=dict(xaxis_title="R, мм", yaxis_title="A, мм", zaxis_title="S, мм",
               aspectmode="data"),
    height=700, margin=dict(l=0, r=0, t=40, b=0))
show(figure, "03_compartments")

## 11. Разница по компартментам

Жир, мышца и кость **несжимаемы** за время задержки дыхания: их объём между фазами обязан
совпасть. Отклонение — это погрешность метода, и меньше неё никакое изменение толщины
обсуждать нельзя. Это единственная проверка во всей работе, у которой ответ известен заранее.

Смещение центра масс говорит о направлении: если стенка и лёгкое едут вместе, изменилась не
толщина, а положение.

In [ ]:
voxel_ml = sx * sy * sz / 1000.0
print(f"{'ткань':>8} {'вдох, мл':>10} {'выдох, мл':>10} {'разница':>9} | "
      f"смещение центра масс, мм: R / A / S")
for name, per_phase in masks.items():
    first, second = per_phase["вдох"], per_phase["выдох"]
    volume_in, volume_ex = first.sum() * voxel_ml, second.sum() * voxel_ml
    shift = (np.array(ndimage.center_of_mass(second))
             - np.array(ndimage.center_of_mass(first))) * np.array([sx, sy, sz])
    print(f"{name:>8} {volume_in:10.0f} {volume_ex:10.0f} "
          f"{100 * (volume_ex - volume_in) / volume_in:+8.1f}% | "
          f"{shift[0]:+6.1f} / {shift[1]:+6.1f} / {shift[2]:+6.1f}")
print()
print("Лёгкие менять объём обязаны. Остальные три — нет, и их отклонение задаёт")
print("пол погрешности всего метода.")

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for axis, (name, per_phase) in zip(axes, masks.items(), strict=True):
    profile_in = per_phase["вдох"].sum(axis=(0, 1)) * sx * sy / 100.0
    profile_ex = per_phase["выдох"].sum(axis=(0, 1)) * sx * sy / 100.0
    z = np.arange(len(profile_in)) * sz
    axis.plot(z, profile_in, color="tab:blue", label="вдох")
    axis.plot(z, profile_ex, color="tab:orange", label="выдох")
    axis.fill_between(z, profile_in, profile_ex, color="tab:red", alpha=0.2)
    axis.set_xlim(BAND[0] * sz, BAND[1] * sz)
    axis.set_title(name)
    axis.set_xlabel("уровень, мм")
    axis.set_ylabel("сечение, см²")
    axis.legend(fontsize=8)
    axis.grid(alpha=0.3)
fig.suptitle("площадь ткани по срезам; закрашено — разница между фазами")
fig.tight_layout()
plt.show()

## 12. Толщина стенки по секторам: обе фазы на одном графике

Тело делится на угловые секторы, и в каждом берётся медиана толщины слоя «жир + мышца».
Три вещи сделаны иначе, чем напрашивается, и каждая чинит конкретную ошибку.

**Центр углов один на обе фазы.** Центр масс тела между вдохом и выдохом уезжает на
десяток миллиметров, и сектор, заданный по своей фазе, называет в каждой разный кусок
стенки. Берётся центр фазы вдоха; после жёсткого совмещения обе фазы в одной системе, так
что это законно.

**Толщина меряется по нормали, а не вдоль радиуса.** Вдоль луча она завышена в
`1/cos α`, где `α` — угол между лучом и нормалью. Грудная клетка сзади плоская, спереди
выпуклая, поэтому завышение зависит от угла и между фазами **не сокращается**. Угол
используется только для группировки.

**Показан разброс.** Ожидаемое изменение порядка миллиметра, а пол погрешности мы измерили
независимо: объём кости и жира за задержку дыхания «меняется» на 3 %, хотя обе несжимаемы.
Медиана без разброса выглядела бы результатом, не будучи им. Полоса — межквартильный размах
разности по срезам внутри сектора.

In [ ]:
SECTOR_DEG = 10  # 360 секторов тоньше анатомии; считаем по 1°, показываем по 10°
LP_FULL = LayerParams(sector_half_angle_deg=180.0, max_walk_mm=60.0)

# Один центр на обе фазы: сектор обязан называть одно и то же место.
middle_slice = int(np.median(shared))
centre_in = ndimage.center_of_mass(body_mask(inhale[:, :, middle_slice], MP))
ORIGIN = (float(centre_in[0]), float(centre_in[1]))

layers = {
    name: measure_layers(volume, spacing, Side.RIGHT, LP_FULL, BAND, ORIGIN)
    for name, volume, _ in PHASES
}
for name, result in layers.items():
    print(f"{name}: {len(result.samples)} отсчётов, "
          f"до ребра дошло {100 * result.bone_fraction:.0f}%")

edges = np.arange(-180, 181, SECTOR_DEG)
centres = 0.5 * (edges[:-1] + edges[1:])


def by_sector(result, field):
    """Медиана толщины в каждом секторе и по срезам внутри него."""
    angle = result.angle_deg
    slices = np.array([item.slice_index for item in result.samples])
    values = getattr(result, field)
    overall, per_slice = [], []
    for low, high in zip(edges[:-1], edges[1:], strict=True):
        inside = (angle >= low) & (angle < high)
        if inside.sum() < 40:
            overall.append(np.nan)
            per_slice.append({})
            continue
        overall.append(float(np.median(values[inside])))
        per_slice.append({
            int(k): float(np.median(values[inside & (slices == k)]))
            for k in np.unique(slices[inside])
            if (inside & (slices == k)).sum() >= 5
        })
    return np.array(overall), per_slice


fields = {"жир + мышца": "soft_mm", "жир": "fat_mm", "мышца": "muscle_mm"}
profile, scatter = {}, {}
for label, field in fields.items():
    profile[label] = {}
    scatter[label] = {}
    for name in layers:
        profile[label][name], scatter[label][name] = by_sector(layers[name], field)

# Разность считается по срезам внутри сектора, чтобы у неё был разброс.
spread = {}
for label in fields:
    lows, mids, highs = [], [], []
    for inhale_slices, exhale_slices in zip(scatter[label]["вдох"],
                                            scatter[label]["выдох"], strict=True):
        common = sorted(set(inhale_slices) & set(exhale_slices))
        if len(common) < 5:
            for target in (lows, mids, highs):
                target.append(np.nan)
            continue
        diff = np.array([exhale_slices[k] - inhale_slices[k] for k in common])
        lows.append(np.percentile(diff, 25))
        mids.append(np.median(diff))
        highs.append(np.percentile(diff, 75))
    spread[label] = (np.array(lows), np.array(mids), np.array(highs))

theta = np.radians(centres)
closed = np.append(theta, theta[0])

fig = plt.figure(figsize=(17, 5.4))
polar = fig.add_subplot(1, 3, 1, projection="polar")
for name, colour in (("вдох", "tab:blue"), ("выдох", "tab:orange")):
    values = profile["жир + мышца"][name]
    polar.plot(closed, np.append(values, values[0]), color=colour, label=name,
               linewidth=1.8)
polar.set_theta_zero_location("N")
polar.set_theta_direction(-1)
polar.set_title("толщина жир + мышца, мм\n0° = вперёд, 90° = вправо", fontsize=10)
polar.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1), fontsize=9)
polar.grid(alpha=0.35)

diff_polar = fig.add_subplot(1, 3, 2, projection="polar")
low, mid, high = spread["жир + мышца"]
diff_polar.fill_between(closed, np.append(low, low[0]), np.append(high, high[0]),
                        color="tab:red", alpha=0.18, label="межквартильный размах")
diff_polar.plot(closed, np.append(mid, mid[0]), color="tab:red", linewidth=1.8,
                label="медиана разности")
diff_polar.plot(closed, np.zeros_like(closed), color="grey", linewidth=0.9)
diff_polar.set_theta_zero_location("N")
diff_polar.set_theta_direction(-1)
diff_polar.set_title("выдох − вдох, мм", fontsize=10)
diff_polar.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)
diff_polar.grid(alpha=0.35)

flat = fig.add_subplot(1, 3, 3)
for label, colour in (("жир", "tab:green"), ("мышца", "tab:purple")):
    low, mid, high = spread[label]
    flat.fill_between(centres, low, high, color=colour, alpha=0.15)
    flat.plot(centres, mid, color=colour, label=label, linewidth=1.6)
flat.axhline(0, color="grey", linewidth=0.9)
flat.set_xlabel("угол, ° (0 = вперёд, 90 = вправо)")
flat.set_ylabel("выдох − вдох, мм")
flat.set_title("раздельно, с межквартильным размахом", fontsize=10)
flat.legend(fontsize=9)
flat.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print()
for label in fields:
    _, mid, _ = spread[label]
    finite = mid[np.isfinite(mid)]
    beyond = np.abs(finite) > 1.0
    print(f"{label:>12}: медиана по секторам {np.median(finite):+5.2f} мм, "
          f"размах {finite.min():+5.2f}..{finite.max():+5.2f}, "
          f"секторов за пределом 1 мм: {beyond.sum()} из {finite.size}")

## 13. Изменение толщины на поверхности тела

То же изменение, что на полярном графике, но нанесённое на саму поверхность. Каждая вершина
окрашена по разности толщины «жир + мышца» в её секторе и на её уровне; шкала симметрична,
красное — стенка на выдохе толще, синее — тоньше.

Соответствие здесь то же приближение: сектор и уровень, а не точка к точке. Настоящего
соответствия для стенки нет ни в одном размеченном наборе, и TRE выше показывает, что
жёсткое совмещение его не даёт.

Смотреть стоит на **связность цвета**. Если пятна крупные и совпадают с анатомией — это
похоже на эффект. Если цвет рассыпан мелкой мозаикой — это разброс, и его величину даёт
таблица под предыдущим графиком: межквартильный размах внутри сектора 3-8 мм при медиане
0-0,7 мм.

In [ ]:
# Карта разности по (уровень, сектор), из тех же данных, что полярный график.
sector_index = np.clip(((layers["вдох"].angle_deg + 180) // SECTOR_DEG).astype(int),
                       0, len(centres) - 1)
delta_map = np.full((inhale.shape[2], len(centres)), np.nan)
for column, (inhale_slices, exhale_slices) in enumerate(
        zip(scatter["жир + мышца"]["вдох"], scatter["жир + мышца"]["выдох"], strict=True)):
    for k in sorted(set(inhale_slices) & set(exhale_slices)):
        delta_map[k, column] = exhale_slices[k] - inhale_slices[k]

# Пробелы затягиваются по углу: соседние секторы одного уровня — ближайшее,
# что есть, и оставлять дыры в раскраске хуже, чем интерполировать вдоль стенки.
for row in range(delta_map.shape[0]):
    line = delta_map[row]
    if np.isfinite(line).sum() >= 3:
        known = np.flatnonzero(np.isfinite(line))
        delta_map[row] = np.interp(np.arange(line.size), known, line[known])

vertex_slice = np.clip(np.round(verts[:, 2] / sz).astype(int), 0, delta_map.shape[0] - 1)
vertex_angle = np.degrees(np.arctan2(verts[:, 1] / sy - ORIGIN[1],
                                     verts[:, 0] / sx - ORIGIN[0]))
vertex_sector = np.clip(((vertex_angle + 180) // SECTOR_DEG).astype(int),
                        0, len(centres) - 1)
intensity = delta_map[vertex_slice, vertex_sector]
inside = np.isfinite(intensity)
print(f"окрашено {100 * inside.mean():.0f}% вершин; "
      f"остальные вне пояса срезов с лёгким")
intensity = np.where(inside, intensity, 0.0)
limit = float(np.nanpercentile(np.abs(intensity[inside]), 95)) or 1.0

figure = go.Figure(go.Mesh3d(
    x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    intensity=intensity, intensitymode="vertex",
    colorscale="RdBu", reversescale=True, cmin=-limit, cmax=limit,
    colorbar=dict(title="выдох − вдох,<br>мм"),
    opacity=1.0, name="стенка", hoverinfo="none"))
figure.update_layout(
    title=f"{CASE}: изменение толщины жир + мышца на поверхности тела "
          f"(шкала ±{limit:.1f} мм)",
    scene=dict(xaxis_title="R, мм", yaxis_title="A, мм", zaxis_title="S, мм",
               aspectmode="data"),
    height=750, margin=dict(l=0, r=0, t=40, b=0))
show(figure, "04_thickness_change")

## 14. Что считать пройденным

1. позвонок даёт пик в центре по лево-правой оси и смещён к краю по передне-задней;
2. сечение правого лёгкого меняется по срезам плавно, без скачков в десятки см²;
3. смещение ориентиров сходится с опубликованным DIR-Lab в пределах примерно миллиметра;
4. объём кости, жира и мышцы между фазами совпадает; расхождение — это пол погрешности;
5. перекрытие позвоночников после совмещения выше 0,7.

Невыполнение пункта 2 означает, что выделение лёгкого не годится и любые толщины на нём
недействительны. Невыполнение пункта 3 означает ошибку в чтении данных, а не в анатомии.
Невыполнение пункта 4 задаёт нижнюю границу того, что вообще можно измерить, и её надо
предъявлять вместе с результатом.